In [ ]:
import os, sys, gc, logging

import pickle as pkl
import numpy as np
os.environ["OMP_MAX_ACTIVE_LEVELS"] = "1"
import numba as nb

from numba import njit, prange
from numba.typed import List
from scipy.ndimage import label as ndi_label
from tqdm import tqdm

---
---
---

In [ ]:
offsets = np.array([[-1, -1, -1], [-1, -1,  0], [-1, -1,  1],
                    [-1,  0, -1], [-1,  0,  0], [-1,  0,  1], 
                    [-1,  1, -1], [-1,  1,  0], [-1,  1,  1],
                    [ 0, -1, -1], [ 0, -1,  0], [ 0, -1,  1],
                    [ 0,  0, -1],               [ 0,  0,  1],
                    [ 0,  1, -1], [ 0,  1,  0], [ 0,  1,  1],
                    [ 1, -1, -1], [ 1, -1,  0], [ 1, -1,  1],
                    [ 1,  0, -1], [ 1,  0,  0], [ 1,  0,  1],
                    [ 1,  1, -1], [ 1,  1,  0], [ 1,  1,  1]])

---
---
---

# General simple functions

We just wrote established python default functions (or knowns onese from mainstream libraries) into numba to speed them up and to make them compatible with the numba functions they are inserted into.

In [ ]:
@njit(fastmath=True)
def isin(a, b):
    
    '''
    Checks if a list of elements is found inside a list of lists of such elements.
    
    Note: The order of the elements must be the same.
    '''
    
    ret = False
    for i in range(len(a)):
        if np.all(b == a[i]):
            ret = True; break
    
    return ret

In [ ]:
@njit(fastmath=True)
def isin_val(a, b):
    
    ret = False
    for i in range(len(a)):
        if b == a[i]:
            ret = True; break
    
    return ret

In [ ]:
@njit(fastmath=True)
def are_neighbors(coord1, coord2, size):
    for offset in offsets:
        if np.array_equal((coord1 + offset) % size, coord2): return True
    return False

In [ ]:
@njit(fastmath=True)
def argwhere_single(fg, vv, size):

    list_argwhere = [[0,0,0]][:0]

    for         i00 in range(size):
        for     j00 in range(size):
            for k00 in range(size):
                if fg[i00][j00][k00] == vv: list_argwhere.append([i00,j00,k00])

    return list_argwhere

---

In [ ]:
@njit(parallel=True, fastmath=True)
def replacement_single(fg, vv, vv0, size):

    for         i00 in nb.prange(size):
        for     j00 in range(    size):
            for k00 in range(    size):
                if fg[i00][j00][k00] == vv: fg[i00][j00][k00] = vv0

In [ ]:
@njit(parallel=True, fastmath=True)
def replacement(fg, void_vals, vv0, size):

    for vv in void_vals:
        if vv != vv0:
            for         i00 in nb.prange(size):
                for     j00 in range(    size):
                    for k00 in range(    size):
                        if fg[i00][j00][k00] == vv: fg[i00][j00][k00] = vv0

---
---
---

# Fast neighbour finders and bridge checkers

In [ ]:
@njit(fastmath=True)
def exterior_outline_pair_maker(cells_group, size):
    
    exterior_outline_pair = [[0,0,0]][:0]
    for cells_group_i in cells_group:
        for j in [(cells_group_i + offset) % np.array([size, size, size]) for offset in offsets]:
            if not isin(cells_group, j):
                if   len(exterior_outline_pair) == 0:              exterior_outline_pair.append(list(j))
                elif not isin(np.array(exterior_outline_pair), j): exterior_outline_pair.append(list(j))
    
    return np.array(exterior_outline_pair)

In [ ]:
@njit(fastmath=True)
def exterior_outline_pair_maker_single(cells_group_i, size):

    exterior_outline_pair = [[0,0,0] for _ in range(26)]
    eop = [(cells_group_i + offset) % np.array([size, size, size]) for offset in offsets]
    for j in range(26): exterior_outline_pair[j] = list(eop[j])
    
    return np.array(exterior_outline_pair)

---

In [ ]:
@njit(fastmath=True)
def check_neighbors_cube(fg, cells_group, size, void_index_od, stop_after_2=False, stop_after_2_regulars=False):
    
    '''
    Finds the neighbors of a cell inside the data cube, where a neighbor is defined as a cell already found to 
        be a void (thus values >=0, see description inside the Finder code).
    
    Note: The np.clip() function and the summation of arrays require arrays (obviously). So the input must also
        be arrays (inside the @njit function).
    '''
    

    # Coordinates of all the cell's neighbors (including over the edge)
    neighbor_coords = exterior_outline_pair_maker(cells_group, size)
    neighbor_values = [0][:0]
    no_regulars = 0
    for nc in range(len(neighbor_coords)):
        if stop_after_2          and len(neighbor_values) == 2: break
        if stop_after_2_regulars and no_regulars          == 2: break
        [i,j,k] = neighbor_coords[nc]
        nc_val = fg[i][j][k]
        if (nc_val >= 0) and ((len(neighbor_values) == 0) or (not isin_val(neighbor_values, nc_val))):
            neighbor_values.append(nc_val)
            if nc_val < void_index_od: no_regulars += 1
    
    return neighbor_values

In [ ]:
@njit(fastmath=True)
def check_neighbors_cube_single(fg, cells_group_i, size, void_index_od, stop_after_2=False, stop_after_2_regulars=False):
    
    '''
    Finds the neighbors of a cell inside the data cube, where a neighbor is defined as a cell already found to 
        be a void (thus values >=0, see description inside the Finder code).
    
    Note: The np.clip() function and the summation of arrays require arrays (obviously). So the input must also
        be arrays (inside the @njit function).
    '''
    

    # Coordinates of all the cell's neighbors (including over the edge)
    neighbor_coords = exterior_outline_pair_maker_single(cells_group_i, size)
    neighbor_values = [0][:0]
    no_regulars = 0
    for nc in range(len(neighbor_coords)):
        if stop_after_2          and len(neighbor_values) == 2: break
        if stop_after_2_regulars and no_regulars          == 2: break
        [i,j,k] = neighbor_coords[nc]
        nc_val = fg[i][j][k]
        if (nc_val >= 0) and ((len(neighbor_values) == 0) or (not isin_val(neighbor_values, nc_val))):
            neighbor_values.append(nc_val)
            if nc_val < void_index_od: no_regulars += 1
    
    return neighbor_values

---

In [ ]:
@njit(fastmath=True)
def check_neighbors_direct(coords, vals, coord_0, size):
    
    coord_diff = np.abs(coords - coord_0)
    
    condition = ((coord_diff[:, 0] <= 1) | (coord_diff[:, 0] == size-1)) & ((coord_diff[:, 1] <= 1) | (coord_diff[:, 1] == size-1)) & ((coord_diff[:, 2] <= 1) | (coord_diff[:, 2] == size-1))
    
    return np.unique(vals[condition])

In [ ]:
@njit(fastmath=True)
def check_walls_direct(coords, coord_0, size):
    
    coord_diff = np.abs(coords - coord_0)

    condition = ((coord_diff[:, 0] <= 1) | (coord_diff[:, 0] == size-1)) & ((coord_diff[:, 1] <= 1) | (coord_diff[:, 1] == size-1)) & ((coord_diff[:, 2] <= 1) | (coord_diff[:, 2] == size-1))

    return len(coords[condition]) != 0

In [ ]:
@njit(fastmath=True)
def check_ghost_bridge(coords, coord_0, fg, set_val, initially_ignore_od, void_index_od, size):

    coord_diff = np.abs(coords - coord_0)

    condition = ((coord_diff[:, 0] <= 1) | (coord_diff[:, 0] == size-1)) & ((coord_diff[:, 1] <= 1) | (coord_diff[:, 1] == size-1)) & ((coord_diff[:, 2] <= 1) | (coord_diff[:, 2] == size-1))
    
    coords = coords[condition]

    found_one = False
    for coord in coords:
        neighbor_coords = (coord + offsets) % np.array([size, size, size])
        neighbor_values = [0][:0]
        
        for [i0,j0,k0] in neighbor_coords:
            fg0 = fg[i0][j0][k0]
            if (fg0 != set_val) and (fg0 >= 0) and (fg0 < void_index_od if initially_ignore_od else True):
                found_one = True; break

        if found_one: break

    return found_one

---

In [ ]:
@njit(fastmath=True)
def find_connected_groups(pairs, pairs_remaining, len_pairs, size):

    # List of the pairs:
    #     -2 not available
    #     -1 not used yet
    #     >=0 index of the pair
    indices_connected = np.full(len_pairs, -2, dtype=np.int64)
    for i0 in range(len_pairs):
        if pairs_remaining[i0]: indices_connected[i0] = -1

    
    index_pair = -1
    # We go pair by pair looking for new groups.
    for i0 in range(len_pairs):
        
        # If this pair has not yet been put in a group, it starts a new one.
        if indices_connected[i0] == -1:
            index_pair += 1
            indices_connected[i0] = index_pair

            # As long as we have a new element in this group, we look if it doesn't allow for new ones.
            queue = List([i0])
            while queue:
                queue_i = queue.pop(0)

                # We look for all pairs beginning at the one that started this group (any others before it
                #     eaither are part of a group already or have started their own).
                for j0 in range(i0+1, len_pairs):
                    if indices_connected[j0] == -1 and are_neighbors(pairs[queue_i], pairs[j0], size):
                        indices_connected[j0] = index_pair
                        queue.append(j0)

    return indices_connected

---
---
---

# Permutations of pair cells

In [ ]:
@njit(fastmath=True)
def factorial(n):

    if n >= 10: n = 10
    result = 1
    for i in range(2, n+1): result *= i
    
    return result

In [ ]:
@njit(fastmath=True)
def permutation(pool, n, perm_index):
    
    '''
    Creates a unique permutation of the elements inside a list, which is replicable thoough the perm_index 
        parameter.
    '''

    
    perm_index -= 1
    
    ran_out_of_perm = False
    if perm_index+1 > factorial(n):
        permt = pool
        ran_out_of_perm = True
        
    if not ran_out_of_perm:
        indices = list(range(n))
        cycles = list(range(n, 0, -1))

        for i0 in range(perm_index):
            for i in range(n-1, -1, -1):
                cycles[i] -= 1
                if cycles[i] == 0:
                    indices[i:] = indices[i+1:] + indices[i:i+1]
                    cycles[i] = n - i
                else:
                    j = cycles[i]
                    indices[i], indices[-j] = indices[-j], indices[i]
                    break
    
        permt = pool[np.array(indices)]
    
    return permt

In [ ]:
@njit(fastmath=True)
def reverse_permutation(permt, n, perm_index):
    
    '''
    Reverses the permutation applied by the permutation function for a given perm_index.
    '''
    
    
    indices = np.arange(n)
    perm_index -= 1
    cycles = np.arange(n, 0, -1)
    
    if perm_index + 1 <= factorial(n):
        for i0 in range(perm_index):
            for i in range(n-1, -1, -1):
                cycles[i] -= 1
                if cycles[i] == 0:
                    indices[i:] = np.concatenate((indices[i+1:], indices[i:i+1]))
                    cycles[i] = n - i
                else:
                    j = cycles[i]
                    indices[i], indices[-j] = indices[-j], indices[i]
                    break
    
    inv_indices = np.zeros(n, dtype=np.int64)
    for i in range(n):
        inv_indices[indices[i]] = i
    
    original = permt[inv_indices]
    
    return original

---
---
---

# ud flood: patches the fast numba way

In [ ]:
# union finder
@njit(cache=True, nogil=True)
def uf_find(parent, x):
    
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    
    return x

In [ ]:
@njit(cache=True, nogil=True)
def uf_union(parent, rank, a, b):
    
    ra = uf_find(parent, a)
    rb = uf_find(parent, b)
    
    if ra == rb: return
    
    if   rank[ra] < rank[rb]: parent[ra] = rb
    elif rank[ra] > rank[rb]: parent[rb] = ra
    else:                     parent[rb] = ra; rank[ra] += 1

In [ ]:
# Periodic merges (26)
@njit(cache=True, nogil=True)
def periodic_merger(labels, parent, rank, size):

    
    # across x
    for     y in range(size):
        for z in range(size):
            l0 = labels[0][y][z]
            if l0 == 0: continue
            
            for     dy in (-1,0,1):
                yy     = (y+dy)%size
                for dz in (-1,0,1):
                    zz = (z+dz)%size
                    
                    l1 = labels[size-1][yy][zz]
                    if (l1 != 0) and (l1 != l0): uf_union(parent, rank, l0, l1)

    
    # across y
    for     x in range(size):
        for z in range(size):
            l0 = labels[x][0][z]
            if l0 == 0: continue
            
            for     dx in (-1,0,1):
                xx     = (x+dx)%size
                for dz in (-1,0,1):
                    zz = (z+dz)%size
                    
                    l1 = labels[xx][size-1][zz]
                    if (l1 != 0) and (l1 != l0): uf_union(parent, rank, l0, l1)

    
    # across z
    for     x in range(size):
        for y in range(size):
            l0 = labels[x][y][0]
            if l0 == 0: continue
            
            for     dx in (-1,0,1):
                xx     = (x+dx)%size
                for dy in (-1,0,1):
                    yy = (y+dy)%size
                    
                    l1 = labels[xx][yy][size-1]
                    if (l1 != 0) and (l1 != l0): uf_union(parent, rank, l0, l1)

In [ ]:
@njit(cache=True, nogil=True)
def compress_all(parent):
    
    for i in range(parent.size): parent[i] = uf_find(parent, i)

In [ ]:
@njit(cache=True, nogil=True)
def build_root_map(n_labels, parent):
    
    root_map = np.empty(n_labels+1, dtype=np.int32)
    for i in range(n_labels+1): root_map[i] = uf_find(parent, i)
    
    return root_map

In [ ]:
# Per-component minima from origins
@njit(cache=True, nogil=True)
def compute_root_minima(labels, root_map, origins_isolated_ud, origins_isolated_ud_index, origins_pairs_ud, origins_pairs_ud_index, mK):

    # basically the largest number that can be represented in int32 - 2**31+1
    # it's not 2**32 because that's the overflow... and +1 because we start from 0
    INF = np.int32(0x7FFFFFFF)
    minima = np.full(root_map.size, INF, dtype=np.int32)


    
    for k in range(origins_isolated_ud.shape[0]):
        x = int(origins_isolated_ud[k][0]); y = int(origins_isolated_ud[k][1]); z = int(origins_isolated_ud[k][2])
        lab = labels[x][y][z]
        if lab == 0: continue
        
        root = root_map[lab]
        v    = origins_isolated_ud_index[k]
        if v < minima[root]: minima[root] = v

    
    if MK == "MK2":
        for k in range(origins_pairs_ud.shape[0]):
            x = int(origins_pairs_ud[k][0]); y = int(origins_pairs_ud[k][1]); z = int(origins_pairs_ud[k][2])
            lab = labels[x][y][z]
            if lab == 0: continue
            
            root = root_map[lab]
            v = origins_pairs_ud_index[k]
            if v < minima[root]: minima[root] = v

    
    return minima

In [ ]:
@njit(cache=True, parallel=True, fastmath=True, nogil=True)
def write_minima_inplace(grid_lvld, labels, root_map, root_minima, size):

    INF = np.int32(0x7FFFFFFF)
    
    for i in prange(size**3):
        x = i // size**2
        r = i %  size**2
        y = r // size
        z = r %  size

        lab = labels[x][y][z]
        if lab == 0:
            grid_lvld[x][y][z] = -1
        else:
            root = root_map[lab]
            m    = root_minima[root]
            if m == INF: grid_lvld[x][y][z] = -1
            else:        grid_lvld[x][y][z] = m

In [ ]:
def ud_patches(grid_lvld, ud_lvl, origins_isolated_ud, origins_isolated_ud_index, origins_pairs_ud, origins_pairs_ud_index, size, MK):
    
    
    # C-accelerated 26c labeling (this doesn't do periodic boundaries... but wwe fix that next)
    mask = (grid_lvld < ud_lvl)
    labels, n = ndi_label(mask, structure=np.ones((3,3,3),dtype=bool))
    del mask; gc.collect()

    # Now we fix the periodic boundaries: via union-find on label IDs
    parent = np.arange(n+1, dtype=np.int32)
    rank   = np.zeros( n+1, dtype=np.uint8)
    periodic_merger(labels, parent, rank, size)
    compress_all(parent)
    root_map = build_root_map(n, parent)
    del parent, rank

    # per-root minima from origins
    root_minima = compute_root_minima(labels, root_map,
                                      origins_isolated_ud, origins_isolated_ud_index,
                                      origins_pairs_ud, origins_pairs_ud_index, MK)

    # final in-place write (>=ud_lvl -> -1; <ud_lvl -> min origin value or -1 if none)
    write_minima_inplace(grid_lvld, labels, root_map, root_minima, size)
    del labels; gc.collect()

---

In [ ]:
@njit(fastmath=True)
def combine_od_connection(old_val, new_val):

    if old_val == -2 or new_val == -2: return -2
    if old_val == -1:                  return new_val
    if new_val == -1:                  return old_val
    if old_val == new_val:             return old_val

    return -2

In [ ]:
@njit(fastmath=True)
def register_od_regular_touch_from_cells(fg, cells_group, od_connections_all_val, size, void_index_od):

    for cell_i in cells_group:

        i, j, k = cell_i
        cell_val = fg[i][j][k]

        if cell_val < 0: continue

        neighb_vals = check_neighbors_cube_single(fg, cell_i, size, void_index_od, stop_after_2=False, stop_after_2_regulars=False)

        # Current cell is regular; register neighbouring od voids.
        if cell_val < void_index_od:
            for neighb_val in neighb_vals:
                if neighb_val >= void_index_od:
                    idx = neighb_val - void_index_od
                    if 0 <= idx < len(od_connections_all_val): od_connections_all_val[idx] = combine_od_connection(od_connections_all_val[idx], cell_val)

        # Current cell is od; register neighbouring regular voids.
        else:
            idx = cell_val - void_index_od
            if 0 <= idx < len(od_connections_all_val):
                for neighb_val in neighb_vals:
                    if 0 <= neighb_val < void_index_od: od_connections_all_val[idx] = combine_od_connection(od_connections_all_val[idx], neighb_val)

---
---
---

# loop_isolated()

In [ ]:
@njit(fastmath=True)
def loop_isolated(fg, isolated_i, od_connections_all_val, pure_isolated, ud, od, void_index_od, size):
    
    '''
    Only looking at the isolated ("alone") cells.
    Because of that, we can instantly modify the filled_grid.
    '''


    len_isolated = len(isolated_i)
    isolated_remaining = np.ones(len_isolated, dtype=np.bool_)
    
    walls_to_be_removed_loop = np.zeros(len_isolated, dtype=np.bool_)

    # The while loop is only needed for MK1, since it is only MK2 that doesn't treat pairs as isolated cells.
    # However, even in MK2 we use this loop for ud pairs. There, we run loop_isolated for each pair.
    try_again = True; first_run = True
    while try_again and (np.sum(isolated_remaining) != 0) and (not pure_isolated or first_run):
        try_again = False; first_run = False
        
        # go thorugh them cell by cell
        for indx_i, isolated_ij in enumerate(isolated_i):
            if isolated_remaining[indx_i]:

                # We set it as checked. So unless we need to look at it again, this is it.
                isolated_remaining[indx_i] = False
                i0,j0,k0 = isolated_ij
                
                # Find all of the cell's (non-wall) neighbors in the filled_grid (including over the edge, ofc).
                void_vals = check_neighbors_cube_single(fg, isolated_ij, size, void_index_od)
                no_voids  = len(void_vals)
    
                
                if no_voids == 1:
                    # Only touches one void (and maybe some walls) -> becomes that void.
                    try_again = True   # MK2 ud pairs.... so only applies if not pure_isolated
                    fg[i0][j0][k0] = void_vals[0]
    
                elif no_voids == 0:
                    if pure_isolated:
                        # If we use MK2 and an isolated cell (so not when we run this in ud for pairs) does not touch
                        #     any void, it was isolated from a previous level... must become a wall.
                        fg[i0][j0][k0] = -3
                        walls_to_be_removed_loop[indx_i] = True
                    else:
                        # If we use MK1 pairs or MK2 ud pairs, we either try_again or come back to this case after.
                        isolated_remaining[indx_i] = True
                        
        
                
                else:
                    # Touches multiple voids (and maybe walls) -> becomes a wall... unless we're ud, then it merges the voids.
                    
                    if ud:
                        # If we are ud, combine all the voids it neighbors into one.
                        try_again = True

                        # Since we may also be in od regime, we need to attempt to pick a non-od void (such that later we do not remove it
                        #    alongside some non-od voids that might now have been merged into it).
                        vv0 = min(void_vals)
                        fg[i0][j0][k0] = vv0
                        # and replace the index of the others neighboring it to its own.
                        replacement(fg, void_vals, vv0, size)
                        # Since we are ud, we can't have any walls yet.... so we don't need to worry about removing any walls between the merged voids.
                        
                        # If we are not in od regime, there are no od voids possible to have merged (since od represents the starting level from which no
                        #     new void origin can be created... but of course we grow these voids as potholes).
                        
                        # But if we are in od regime, it might seem like is time to complicate things, but really, no previous od connections could have 
                        #     been formed by now (since we would have had to have been in ud regime, so they all resulted in mergers) and so the only thing
                        #     we had to have done is the above: to merge them!
                        # Ok... but then does it even matter if od < ud? Yes! Because od sets the level from which no new void can be formed and so, if one
                        #     od does not merge in the ud regime (it survives), it would be survive till the end! So yes, it does matter!
                        
                    
                    else:
                        # Again, it's not that we can't be in od regime while also under ud regime.
                        # It's simply that if we are in ud, there can be no voids and we merge all which we encounter
                        if not od:
                            # If we are not od, we don't need to make pairs for the merged voids.
                            fg[i0][j0][k0] = -2
                        
                        else:
                            # But if we are... we do.
                            regular_neighbors = -1; regular_neighbors_len = 0; od_neighbors = [0][:0]
                            for iii in void_vals:
                                if iii < void_index_od:
                                    regular_neighbors = iii; regular_neighbors_len += 1
                                else:
                                    od_neighbors.append(iii)
                            
                            if   regular_neighbors_len >= 2:
                                # If there are at least two values that are non-od voids, it is a wall.
                                # So this also won't form a connection between od's.
                                fg[i0][j0][k0] = -2

                            
                            else:
                                # If it only touches one or no regulars, then it must also neighbor at least one (or necessarily
                                #     multiple, respectively) od('s).
                                try_again = True
                                
                                if regular_neighbors_len == 1:
                                    # If it only touches one regular, we glue the cell to it...
                                    fg[i0][j0][k0] = regular_neighbors
                                    
                                    # and set the (first) od as conected to it.
                                    odn0 = od_neighbors[0]
                                    od_connections_all_val[odn0-void_index_od] = combine_od_connection(od_connections_all_val[odn0-void_index_od], regular_neighbors)

                                else:
                                    # If it touches no regulars, then we attribute this cell to the first od.
                                    fg[i0][j0][k0] = od_neighbors[0]


                                # If we had multiple od's...
                                if len(od_neighbors) > 1:
                                    # we must merge them all into the first one...
                                    replacement(fg, od_neighbors, od_neighbors[0], size)

                                    # and append all their connections to it accordingly.
                                    odn0   = od_neighbors[0]
                                    odcav0 = od_connections_all_val[odn0-void_index_od]
                                    
                                    for odn in od_neighbors[1:]:
                                        odcav = od_connections_all_val[odn-void_index_od]
                                        odcav0 = combine_od_connection(odcav0, odcav)
                                        od_connections_all_val[odn-void_index_od] = -1
                                    
                                    od_connections_all_val[odn0-void_index_od] = odcav0


    if not pure_isolated:   # otherwise we already did this
        for indx_i, isolated_ij in enumerate(isolated_i):
            if isolated_remaining[indx_i]:
                i0,j0,k0 = isolated_ij
                fg[i0][j0][k0] = -3
                walls_to_be_removed_loop[indx_i] = True


    return [list(_) for _ in isolated_i[walls_to_be_removed_loop]]

---

In [ ]:
@njit(fastmath=True)
def loop_isolated_leftovers(fg, isolated_i, size, void_index_od):

    
    len_isolated = len(isolated_i)
    isolated_remaining       = np.ones(len_isolated, dtype=np.bool_)
    
    try_again = True
    while try_again and (np.sum(isolated_remaining) != 0):
        try_again = False
        
        # go thorugh them cell by cell
        for indx_i, isolated_ij in enumerate(isolated_i):
            if isolated_remaining[indx_i]:

                # We set it as checked. So unless we need to look at it again, this is it.
                isolated_remaining[indx_i] = False
                i0,j0,k0 = isolated_ij
                
                # Find all of the cell's (non-wall) neighbors in the filled_grid (including over the edge, ofc).
                # Again, since these od can only neighbour a regular and not an od (they would have been mergerd into this one by now), we
                #     do not need to worry about anything except the number of neighbours.
                void_vals = check_neighbors_cube_single(fg, isolated_ij, size, void_index_od, stop_after_2=True)
                no_voids  = len(void_vals)
                
                if no_voids == 1:
                    try_again = True
                    fg[i0][j0][k0] = void_vals[0]
    
                elif no_voids == 0:
                    isolated_remaining[indx_i] = True
                        
                elif no_voids >= 2:
                    fg[i0][j0][k0] = -2


    # It may be that when we run this od void as a large pair (interpretted as isolated cells in MK1), we create a region that
    #     falsely isolates some of its cells.
    # This will be dealt with in the thinning process, but, for now, we must set them as -3's.
    walls_to_be_removed_loop = np.zeros(len_isolated, dtype=np.bool_)
    for indx_i, isolated_ij in enumerate(isolated_i):
        if isolated_remaining[indx_i]:
            i0,j0,k0 = isolated_ij
            fg[i0][j0][k0] = -3
            walls_to_be_removed_loop[indx_i] = True


    return [list(_) for _ in isolated_i[walls_to_be_removed_loop]]

---
---
---

# loop_pairs()

In [ ]:
@njit(fastmath=True)
def loop_pairs(fg, pairs, od_connections_all_val, size, initially_ignore_od, void_index_od, times_we_tried_max=100):
    
    
    
    len_pairs = len(pairs)
    pairs_remaining = np.ones(len_pairs, dtype=np.bool_)

    permutations_done = 1
    times_we_tried = 0; try_it_all_again = True
    
    while try_it_all_again:
        try_it_all_again = False
        
        walls_to_be_removed_loop = np.zeros(len_pairs, dtype=np.bool_)
        
        times_we_tried += 1

        # In case we try a different permuation... we remove all the ghosts and super ghosts from the grid.
        all_gsg_vw_coord = np.zeros(len_pairs, dtype=np.bool_)
        
        len_pairs_remaining = np.sum(pairs_remaining)
        less_than_4_left = len_pairs_remaining < 4
        
        ##########################################################################################
        
        
        # Gotta define them here so numba won't be mad at us.
        p_c_v_coord  = np.zeros(len_pairs, dtype=np.bool_)
        p_g_v_coord  = np.zeros(len_pairs, dtype=np.bool_)
        p_sg_v_coord = np.zeros(len_pairs, dtype=np.bool_)
        p_c_v_vals   = np.zeros(len_pairs, dtype=np.int64)
        p_g_v_vals   = np.zeros(len_pairs, dtype=np.int64)
        p_sg_v_vals  = np.zeros(len_pairs, dtype=np.int64)
        
        p_c_w_coord  = np.zeros(len_pairs, dtype=np.bool_)
        p_g_w_coord  = np.zeros(len_pairs, dtype=np.bool_)
        p_sg_w_coord = np.zeros(len_pairs, dtype=np.bool_)

        found_one_g_w   = False; found_one_sg_w  = False
        found_one_p_c_v = False; found_one_p_g_v = False; found_one_p_sg_v = False
        found_one_p_c_w = False; found_one_p_g_w = False; found_one_p_sg_w = False
    
    
        # We try until all the elemnts either touched a void or we ran them once more and none did.
        try_again = True; run = -1
        while try_again and (np.sum(pairs_remaining) != 0):
            try_again = False; run += 1
            
            c_c_v_coord  = np.zeros(len_pairs, dtype=np.bool_)
            c_g_v_coord  = np.zeros(len_pairs, dtype=np.bool_)
            c_sg_v_coord = np.zeros(len_pairs, dtype=np.bool_)
            c_c_v_vals   = np.zeros(len_pairs, dtype=np.int64)
            c_g_v_vals   = np.zeros(len_pairs, dtype=np.int64)
            c_sg_v_vals  = np.zeros(len_pairs, dtype=np.int64)
    
            c_c_w_coord  = np.zeros(len_pairs, dtype=np.bool_)
            c_g_w_coord  = np.zeros(len_pairs, dtype=np.bool_)
            c_sg_w_coord = np.zeros(len_pairs, dtype=np.bool_)
    
            found_one_c_c_v = False; found_one_c_g_v = False; found_one_c_sg_v = False
            found_one_c_c_w = False; found_one_c_g_w = False; found_one_c_sg_w = False
            
            no_pairs_deleted = 0
            for i0 in range(len_pairs):
                if pairs_remaining[i0]:
                    pair_0 = pairs[i0]
                    
                    certain = False; ghost = False; superghost = False; void = False; wall = False
                    use__p_g_v = False; cant_be_super = True
                    set_val = -1
        
        
        
                    #############################################

                    # The first iteration over the pair.
                    if run == 0:
                        # Find all the neighbors of this cell that are voids.
                        # !!! To keep the hyerarchy of peeling the pair, we only look in the original fg: we do not yet modify it
                        #     until the end of the whole first iteration.
                        
                        # Can't just stop after finding two, because some may be od's and there may also be regulars to be found,
                        #     leading to a change in result if initially_ignore_od... so we can still stop after 2 regulars.
                        vals_fg = check_neighbors_cube_single(fg, pair_0, size, void_index_od, stop_after_2=False, stop_after_2_regulars=True)
                        if initially_ignore_od: vals_fg = [i for i in vals_fg if i < void_index_od]
                        len_fg = len(vals_fg)
        
                        if len_fg >= 2:
                            # If we found two voids, ofc it is a certain wall.
                            certain = True; wall = True
                        
                        elif len_fg == 1:
                            # If we only found one, however, to save time, we do look inside this iteration for other certain voids
                            #     to see if it is a certain wall.
                            # (If it isn't, it still has set_val != -1, so we later do check for current ghost voids and walls. If we
                            #     wouldn't, in the next runs we would still have to consider cells that neighbor original fg voids: we
                            #     wouldn't have finished the first peel to only advance based solely on it.)
                            set_val = vals_fg[0]
                            if found_one_c_c_v:
                                vals_c_c_v = check_neighbors_direct(pairs[c_c_v_coord], c_c_v_vals[c_c_v_coord], pair_0, size)
                                len_c_c_v = len(vals_c_c_v)
                                
                                if (len_c_c_v >= 2) or ((len_c_c_v == 1) and vals_c_c_v[0] != set_val):
                                    certain = True; wall = True; set_val = -1
        
                    

                    # The following iterations over the pair.
                    # From now on, the only connections we can find are from the previous peel. This is also a major time-saver, as we
                    #     do not have to check in the original fg.
                    else:
                        if found_one_p_c_v:
                            vals_p_c_v = check_neighbors_direct(pairs[p_c_v_coord], p_c_v_vals[p_c_v_coord], pair_0, size)
                            len_p_c_v = len(vals_p_c_v)
                            use__p_g_v = True
                            
                            if len_p_c_v >= 2:
                                certain = True; wall = True
                            
                            elif len_p_c_v == 1:
                                set_val = vals_p_c_v[0]
                                if found_one_c_c_v:
                                    vals_c_c_v = check_neighbors_direct(pairs[c_c_v_coord], c_c_v_vals[c_c_v_coord], pair_0, size)
                                    len_c_c_v = len(vals_c_c_v)
                
                                    if (len_c_c_v >= 2) or ((len_c_c_v == 1) and (vals_c_c_v[0] != set_val)):
                                        certain = True; wall = True; set_val = -1
                
                                if (not certain) and (found_one_p_sg_v or found_one_c_sg_v):
                                    # Hey kids, wanna see a magic trick?
                                    # Since we were careful that we never set overlapping v/w nor c/g/sg, we can sum these lists.
                                    vals_pc_sg_v = check_neighbors_direct(pairs[p_sg_v_coord+c_sg_v_coord], (p_sg_v_vals+c_sg_v_vals)[p_sg_v_coord+c_sg_v_coord], pair_0, size)
                                    len_pc_sg_v = len(vals_pc_sg_v)
            
                                    if (len_pc_sg_v >= 2) or ((len_pc_sg_v == 1) and (vals_pc_sg_v[0] != set_val)):
                                        ghost = True; wall = True; set_val = -1
        
                                    elif (len_pc_sg_v == 1) and (vals_pc_sg_v[0] == set_val):
                                        ghost = True
                                                
                                        
                                    
                        elif found_one_p_g_v:
                            vals_p_g_v = check_neighbors_direct(pairs[p_g_v_coord], p_g_v_vals[p_g_v_coord], pair_0, size)
                            len_p_g_v = len(vals_p_g_v)
                            
                            if len_p_g_v >= 2:
                                wall = True
        
                                superghost = True
                                if found_one_c_c_v:
                                    vals_c_c_v = check_neighbors_direct(pairs[c_c_v_coord], c_c_v_vals[c_c_v_coord], pair_0, size)
                                    len_c_c_v = len(vals_c_c_v)
                
                                    if   (len_c_c_v >= 2): certain = True; superghost = False
                                    elif (len_c_c_v == 1): ghost   = True; superghost = False
                                    
        
                            
                            elif len_p_g_v == 1:
                                set_val = vals_p_g_v[0]
                                found_one_neighb_c_c_v = False
                                
                                if found_one_c_c_v:
                                    vals_c_c_v = check_neighbors_direct(pairs[c_c_v_coord], c_c_v_vals[c_c_v_coord], pair_0, size)
                                    len_c_c_v = len(vals_c_c_v)
        
                                    if (len_c_c_v != 0):
                                        found_one_neighb_c_c_v = True
                
                                        if (len_c_c_v >= 2):
                                            certain = True; wall = True; set_val = -1
                                        
                                        elif ((len_c_c_v == 1) and (vals_c_c_v[0] != set_val)):
                                            ghost = True; wall = True; set_val = -1
                                        
                                        elif ((len_c_c_v == 1) and (vals_c_c_v[0] == set_val)) and (found_one_p_sg_v or found_one_c_sg_v):
                                            vals_pc_sg_v = check_neighbors_direct(pairs[p_sg_v_coord+c_sg_v_coord], (p_sg_v_vals+c_sg_v_vals)[p_sg_v_coord+c_sg_v_coord], pair_0, size)
                                            len_pc_sg_v = len(vals_pc_sg_v)
                        
                                            if (len_pc_sg_v >= 2) or ((len_pc_sg_v == 1) and (vals_pc_sg_v[0] != set_val)):
                                                ghost = True; wall = True; set_val = -1
                    
                                            elif (len_pc_sg_v == 1) and (vals_pc_sg_v[0] == set_val):
                                                ghost = True
            
                                # If we either found no c_c_v or we did but none touch this cell...
                                if (not found_one_c_c_v) or (not found_one_neighb_c_c_v):
                                    cant_be_super = False
                                    
                                    if (found_one_p_sg_v or found_one_c_sg_v):
                                        vals_pc_sg_v = check_neighbors_direct(pairs[p_sg_v_coord+c_sg_v_coord], (p_sg_v_vals+c_sg_v_vals)[p_sg_v_coord+c_sg_v_coord], pair_0, size)
                                        len_pc_sg_v  = len(vals_pc_sg_v)
                
                                        if (len_pc_sg_v >= 2) or ((len_pc_sg_v == 1) and (vals_pc_sg_v[0] != set_val)):
                                            superghost = True; wall = True; set_val = -1
            
                                        elif (len_pc_sg_v == 1) and (vals_pc_sg_v[0] == set_val):
                                            superghost = True
                                            
                                            
        
                                        
        
                        elif found_one_p_sg_v:
                            vals_p_sg_v = check_neighbors_direct(pairs[p_sg_v_coord], p_sg_v_vals[p_sg_v_coord], pair_0, size)
                            len_p_sg_v = len(vals_p_sg_v)
                            
                            if len_p_sg_v >= 2:
                                wall = True
        
                                superghost = True   # see if it remains after below
                                if found_one_c_c_v:
                                    vals_c_c_v = check_neighbors_direct(pairs[c_c_v_coord], c_c_v_vals[c_c_v_coord], pair_0, size)
                                    len_c_c_v = len(vals_c_c_v)
                
                                    if   (len_c_c_v >= 2): certain = True; superghost = False
                                    elif (len_c_c_v == 1): ghost   = True; superghost = False
                                
        
                            elif len_p_sg_v == 1:
                                set_val = vals_p_sg_v[0]
                                superghost = True   # see if it remains after below
                                if found_one_c_c_v:
                                    vals_c_c_v = check_neighbors_direct(pairs[c_c_v_coord], c_c_v_vals[c_c_v_coord], pair_0, size)
                                    len_c_c_v = len(vals_c_c_v)
                
                                    if (len_c_c_v >= 2):
                                        certain = True; wall = True; superghost = False; set_val = -1
                                    
                                    elif ((len_c_c_v == 1) and (vals_c_c_v[0] != set_val)):
                                        ghost   = True; wall = True; superghost = False; set_val = -1
                                    
                                    elif ((len_c_c_v == 1) and (vals_c_c_v[0] == set_val)):
                                        ghost   = True;              superghost = False
        
        
                                if (not wall):
                                    if found_one_c_sg_v:
                                        vals_c_sg_v = check_neighbors_direct(pairs[c_sg_v_coord], c_sg_v_vals[c_sg_v_coord], pair_0, size)
                                        len_c_sg_v = len(vals_c_sg_v)
        
                                        if (len_c_sg_v >= 2) or ((len_c_sg_v == 1) and (vals_c_sg_v[0] != set_val)):
                                            wall = True
        
                    
                    
                    ######################################################################################
        
        
                    
                    # If we are set to have found a void or a wall...
                    if set_val != -1:
        
                        ##### BLUE
                        # If we are still not sure what type of structure this cell is... and we are not run run==0 (simply because
                        #     in this first run we can't have any super ghosts)...
                        # A new type of check appears: our cell might neighbor a super ghost wall, which can turn it into a ghost or
                        #     even a super ghost if it has not (and thus will not) touch a certain void.
                        # In the later case, if it does not touch a super ghost wall, then it must be a ghost.
                        if (not run == 0) and (not certain) and (not ghost) and (not superghost):
                            
                            coord_to_check  = pairs[p_sg_w_coord + c_sg_w_coord]
        
                            # Has touched a certain void... can't be a super ghost.
                            if cant_be_super:
                                if len(coord_to_check) != 0:
                                    if check_walls_direct(coord_to_check, pair_0, size):
                                        ghost = True
                            
                            # Has not touched a certain void... can be a super ghost, but at least must be a ghost.
                            else:
                                ghost = True
                                if len(coord_to_check) != 0:
                                    if check_walls_direct(coord_to_check, pair_0, size):
                                        superghost = True; ghost = False
                                
                                    
                        
                        ##### PURPLE
                        # If we tried all the above checks...
                        # and still haven't set on wether we find a void or a wall...
                        # we must check ghost voids.
                        if (not void) and (not wall):
                            
                            void = True
                            # If we are guaranteed a different neighboring void, this cell is a wall... otherwise it must be a void.
                            if found_one_p_g_v or found_one_c_g_v:
                            
                                # Check previous (if not included in our above search already) and current voids.
                                coord_to_check = pairs[c_g_v_coord]
                                vals_to_check = c_g_v_vals[c_g_v_coord]
                                if use__p_g_v:
                                    coord_to_check = pairs[c_g_v_coord+p_g_v_coord]
                                    vals_to_check  = (c_g_v_vals+p_g_v_vals)[c_g_v_coord+p_g_v_coord]
    
                                if len(coord_to_check) != 0:
                                    vals_checked = check_neighbors_direct(coord_to_check, vals_to_check, pair_0, size)
                                    len_vals_checked = len(vals_checked)
                                
                                    if (len_vals_checked >= 2) or ((len_vals_checked == 1) and (vals_checked[0] != set_val)):
                                        wall = True; void = False
                                        
                                        # If we arrived to this point, we cannot have a certain wall...
                                        # but in some cases, we don't even know if this is a ghost or a superghost wall.
                                        # Then, since super ghosts require certainty (we would have had to have already touched a super ghost),
                                        #     we know that this must be a ghost.
                                        if (not superghost): ghost = True
        
        
                        ##### GREEN
                        # If we are set to have found a void and we're still not sure what type it can either be a certain or a ghost one.
                        if (void) and (not certain) and (not ghost) and (not superghost):
        
                            # Check all cells that might form a bridge to a previous peel void:
                            # (all remaining pairs... any we already ran over would have already given that bridge in the previous
                            #     checks) + (previous and current walls that are either ghosts or superghosts... so that they could
                            #     have returned a void in another permuation)

                            pairs_remaining_afteri0 = pairs_remaining.copy()
                            for ppi in range(0, i0+1):
                                pairs_remaining_afteri0[ppi] = False
                            coord_to_check = pairs[pairs_remaining_afteri0+p_g_w_coord+p_sg_w_coord+c_g_w_coord+c_sg_w_coord]
        
                            certain = True
                            # If we have such cells to check...
                            if len(coord_to_check) != 0:
                                # and they do form a bridge...
                                if check_ghost_bridge(coord_to_check, pair_0, fg, set_val, initially_ignore_od, void_index_od, size):
                                    ghost = True; certain = False
                        
                    
                    ######################################################################################
        
                    
                    if void:
                        try_again = True
                    if void or wall:
                        pairs_remaining    [i0]   = False
                    
                    
                    if certain:
                        if wall:
                            found_one_c_c_w       = True
                            c_c_w_coord[    i0]   = True
                        else:
                            found_one_c_c_v       = True
                            c_c_v_coord[    i0]   = True; c_c_v_vals[ i0] = set_val
        
                    if ghost:
                        if wall:
                            found_one_c_g_w       = True
                            c_g_w_coord[     i0]  = True
                            all_gsg_vw_coord[i0]  = True
                        else:
                            found_one_c_g_v       = True
                            c_g_v_coord[i0]       = True; c_g_v_vals[ i0] = set_val
                            all_gsg_vw_coord[i0]  = True
        
                    if superghost:
                        if wall:
                            found_one_c_sg_w      = True
                            c_sg_w_coord[    i0]  = True
                            all_gsg_vw_coord[i0]  = True
                        else:
                            found_one_c_sg_v      = True
                            c_sg_v_coord[    i0]  = True; c_sg_v_vals[i0] = set_val
                            all_gsg_vw_coord[i0]  = True
    


            ##########################################################################################


            found_one_p_c_v  = False
            found_one_p_c_w  = False
            found_one_p_g_v  = False
            found_one_p_g_w  = False
            found_one_p_sg_v = False
            found_one_p_sg_w = False
            if found_one_c_c_v:  found_one_p_c_v  = True
            if found_one_c_c_w:  found_one_p_c_w  = True
            if found_one_c_g_v:  found_one_p_g_v  = True
            if found_one_c_g_w:  found_one_p_g_w  = True; found_one_g_w  = True
            if found_one_c_sg_v: found_one_p_sg_v = True
            if found_one_c_sg_w: found_one_p_sg_w = True; found_one_sg_w = True
            
            p_c_v_coord = c_c_v_coord.copy(); p_g_v_coord = c_g_v_coord.copy(); p_sg_v_coord = c_sg_v_coord.copy()
            p_c_v_vals  = c_c_v_vals.copy();  p_g_v_vals  = c_g_v_vals.copy();  p_sg_v_vals  = c_sg_v_vals.copy()
            p_c_w_coord = c_c_w_coord.copy(); p_g_w_coord = c_g_w_coord.copy(); p_sg_w_coord = c_sg_w_coord.copy()
            
            # To make the check_cross_bridge() work, we need at the end of each peel to add to fg all the cells we just found.
            if found_one_c_c_v:
                for i00 in range(len_pairs):
                    if c_c_v_coord[i00]:
                        ip,jp,kp = pairs[i00]
                        fg[ip][jp][kp] = c_c_v_vals[i00]

            if found_one_c_g_v:
                for i00 in range(len_pairs):
                    if c_g_v_coord[i00]:
                        ip,jp,kp = pairs[i00]
                        fg[ip][jp][kp] = c_g_v_vals[i00]

            if found_one_c_sg_v:
                for i00 in range(len_pairs):
                    if c_sg_v_coord[i00]:
                        ip,jp,kp = pairs[i00]
                        fg[ip][jp][kp] = c_sg_v_vals[i00]

            if found_one_c_c_w:
                for i00 in range(len_pairs):
                    if c_c_w_coord[i00]:
                        ip,jp,kp = pairs[i00]
                        fg[ip][jp][kp] = -2

            if found_one_c_g_w:
                for i00 in range(len_pairs):
                    if c_g_w_coord[i00]:
                        ip,jp,kp = pairs[i00]
                        fg[ip][jp][kp] = -2

            if found_one_c_sg_w:
                for i00 in range(len_pairs):
                    if c_sg_w_coord[i00]:
                        ip,jp,kp = pairs[i00]
                        fg[ip][jp][kp] = -2

                    
                    
            
        ##############################################################################################
            

        

        # Since we started from a pair, it cannot be that only some of its cells were isolated from the previous levels. If that were, the whole pair
        #     had to be isolated (obviously) and we would not have have started the loop_pair().
        
        # However, there are other ways some cells can end-up isolated at this stage:
        # 1. It may be that another permutation does not lead to that.
        # 2. We simply formed walls (wether certain/ghosts/superghosts) that isolate them.
        #        This cannot be fixed as it is intrinsic to the fg configuration of this level.
        #        We would have to go back to the previous level and try a different permutation and repeat the process, until, if all permutations
        #            fail, we would then have to go back to the next previous level and repeat this process indefinitely, until we may finally
        #            find that it is simply an intrinsic blocade.
        #        For obvious reasons, this would explode the computational time with very little added benefit for our purposes.
        #        If it is mission critical for such an iteration to occur, we encourage the next users to implement it by passing a new parameter
        #            from this function and implementing the trackback mechanism in the level loop.
        
        len_pairs_modified = np.sum(pairs_remaining)   # pairs we now modify, i.e. the remaining ones
        if len_pairs_modified != 0:

            # First, we then favour finding connections with od's rather than try_it_all_again.
            # Reason being, if we attribute the remaining to od's the left cells, we will still give them back to the regular voids later when the ods merge.
            # This way, by trading a possible marginal improvement (the major one came from the peeling method, which is still present in the od merger) we make
            #     major savings on the computational time.
            # If even one cell remains unfilled, then we do try_it_all_again if we can (we determine the conditions for that afterwards).
            
            
            merge_ods_nvodu = [[0]][:0]
            
            # Of course, the od check only makes sense if we are in the od regime and if the original pair did neighbor some od's.
            all_leftovers_touch_od = False
            if initially_ignore_od:
                all_leftovers_touch_od = True
                
                fcg = find_connected_groups(pairs, pairs_remaining, len_pairs, size)
                for lp_i in range(max(fcg)+1):
                    
                    leftovers_pair = np.zeros((len_pairs, 3), dtype=pairs.dtype)
                    index = 0
                    for i in range(len_pairs):
                        if fcg[i] == lp_i:
                            leftovers_pair[index] = pairs[i]
                            index += 1
                    leftovers_pair = leftovers_pair[:index]
                    
                    # Again, we cannot stop after just 2 od's because we will have to replace them if this is successful.
                    neighb_vals_unique = check_neighbors_cube(fg, leftovers_pair, size, void_index_od)
                    
                    neighb_vals_od_unique = [0][:0]
                    for neighb_val in neighb_vals_unique:
                        if neighb_val >= void_index_od: neighb_vals_od_unique.append(neighb_val)
                    merge_ods_nvodu.append(neighb_vals_od_unique)
                    
                    # If they end-up not touching any od either, we're done.
                    if len(neighb_vals_od_unique) == 0: all_leftovers_touch_od = False
                
                    


            # If od's did not save the day, we try_it_all_again.
            if (not initially_ignore_od) or (not all_leftovers_touch_od):

                # But we can only do this if:
                #     1. We did not reach the maximum number of permutations we allowed ourselves to try.
                #     2. We did not reach the maximum number of permutations possible from the number of cells in this pair.
                #     3. We did find some ghosts/superghosts to backtrack on.
                #     4. We started this last try_it_all_again loop with a pair having at least 4 cells.
                #            - if == 1  --  It was isolated from the start... nothing we can do... must be a "false" -3 wall.
                #            - If we have at least 4, it can be that we have the following connections: (1,2), (3,2) and (3,4) and only
                #                  1 and 2 are connected to a void and these two are different ones. If our order is (1,3,2,4) or (1,3,4,2),
                #                  then in the frist peel we only make 1 and 3 become two different void cells. So, in the second/third,
                #                  respectively, run, 2 will become a wall and isolate 4.
                #                  This process is not possible with less than 4 cells, as it requires one cell to be isolated, one to isolate
                #                      it by becoming a wall and at least two to become different voids to turn it into a wall.

                # If we meet those conditions, then we simply remove the ghosts and superghosts and see if they now can become proper values.
                # If we already did try that (so we ended up with the same number of leftovers as we began with), then we must try a different permutaiton.
                # (It may seem troubling to think that we did not account for the case where the first try_it_all_again run leads to no assigned cells, but that
                #     simply cannot be the case, as we ostarted loop_pairs() knowing some cells do neighbor voids.)
                if (times_we_tried < times_we_tried_max) and (permutations_done <= factorial(len_pairs_modified)) and (found_one_g_w or found_one_sg_w) and (not less_than_4_left):

                    try_it_all_again = True

                    # Again, if this is the first run, of course len_pairs_modified < len_pairs_remaining (remaaining pairs < start of the permutation pairs), as 
                    #     we know the pair connects to some voids.
                    # But for any run, since in the last try of this permutation we did have the ghosts and superghosts, performing the same permutation again
                    #     with them now removed is indeed akin to a new permutation, so we do not need to shuffle the order.
                    if len_pairs_modified < len_pairs_remaining:
                        permutations_done = 1

                    # If we didn't manage to do a thing, we try a different permutation (as long as we tried not too many times and we have (super)ghosts to hopefully change).
                    else:
                        permutations_done += 1
                        
                        # If we're set to try a different permutation, we create a new pair lists consisting of all the remaining
                        #     cells and all the ghosts and super ghosts. We also must remove those from the grid.
                        # Reset the values in the list for all the ghost and superghost voids and walls.
                        for i00, [i0,j0,k0] in enumerate(pairs):
                            if all_gsg_vw_coord[i00]:
                                fg[i0][j0][k0] = -1
                                pairs_remaining[ i00] = True
                                all_gsg_vw_coord[i00] = False
                        
                        # If this isn't the first permutation, we reverse the previous one to get back to the original.
                        if permutations_done != 2:
                            pairs           = reverse_permutation(pairs          , len_pairs, permutations_done-1)
                            pairs_remaining = reverse_permutation(pairs_remaining, len_pairs, permutations_done-1)

                        # And now we perform the next permutation.
                        pairs               =         permutation(pairs          , len_pairs, permutations_done  )
                        pairs_remaining     =         permutation(pairs_remaining, len_pairs, permutations_done  )


                
                # But if we cannot try_it_all_again (we did not meet at least one of the 4 conditions)...
                else:
                    # and we also did not have od's to save us...
                    if not initially_ignore_od:
                        # we're out of options: they're improper walls and no more try_it_all_again.
                        for pp_i, [ip,jp,kp] in enumerate(pairs):
                            if pairs_remaining[pp_i]:
                                fg[ip][jp][kp] = -3
                                walls_to_be_removed_loop[pp_i] = True
                        
                        
                    
            # And yet... if we cannot try_it_again (either because we did manage to find od's for all the leftover connected pairs or because we did not
            #     but we also tried too many tiems or we didn't find any ghosts/superghosts last time) and we also did initially_ignore_od's (because if
            #     we didn't, then we dealt with that case just above).... then we take the leftover connected pairs and attribute them to their respective
            #     od voids and merge them into one.
            if (not try_it_all_again) and initially_ignore_od:

                for i0 in range(max(fcg)+1):
                    
                    leftovers_pair      = np.zeros((len_pairs, 3), dtype=pairs.dtype)
                    leftovers_pair_indx = [0][:0]
                    index = 0
                    for i in range(len_pairs):
                        if fcg[i] == i0:
                            leftovers_pair[index] = pairs[i]
                            leftovers_pair_indx.append(i)
                            index += 1
                    leftovers_pair = leftovers_pair[:index]
                    
                    neighb_vals_od_unique = merge_ods_nvodu[i0]


                    # If this connected group (i.e. leftovers_pair) does not neighbor any od's, then it is isolated and must become a wall.
                    if len(neighb_vals_od_unique) == 0:
                        for pp_i in range(index):
                            ip, jp, kp = leftovers_pair[     pp_i]
                            pp_j       = leftovers_pair_indx[pp_i]
                            fg[ip][jp][kp] = -3
                            walls_to_be_removed_loop[pp_j] = True

                    
                    # But if it touches at least one od void, it is part of the first one.
                    else:
                        nvodu0 = neighb_vals_od_unique[0]
                        for [ip,jp,kp] in leftovers_pair: fg[ip][jp][kp] = nvodu0
                        
                        odcav0 = od_connections_all_val[nvodu0-void_index_od]
                        
                        # If it touches other OD's, merge the rest into the first and give it their connections.
                        if len(neighb_vals_od_unique) >= 2:
                        
                            replacement(fg, np.array(neighb_vals_od_unique), nvodu0, size)
                        
                            for nvodu in neighb_vals_od_unique[1:]:
                        
                                odcav = od_connections_all_val[nvodu-void_index_od]
                                odcav0 = combine_od_connection(odcav0, odcav)
                        
                                od_connections_all_val[nvodu-void_index_od] = -1
                        
                        od_connections_all_val[nvodu0-void_index_od] = odcav0

    
    return  [list(_) for _ in pairs[walls_to_be_removed_loop]]

---
---
---

# thin_walls()

In [ ]:
@njit(fastmath=True)
def thin_walls(fg, walls_to_be_removed, size, void_index_od):
    
    '''
    There is, however, one correction ot the data we shall make.

    We have establighed that there might be wall cells which shouldn't be. This might create the thick walls.
    
    We can partly mitigete this issue by doing the following (remember, much like adding walls, thinning them must also be cone in a layered manner):
        1. Iterate over all the wall cells in this level.
        2. If a cell is a wall and has a single neighbouring void (if it has none, it is fully isolated and we'll fix it in a following round if by then it will only have gained a single void as a neighbor) make that cell part of that void.
        3. Go through the whole level, then repeat until there are no newly set void cells.
        
    Since the cells in the ll_i list are randomly ordered, this is an efficient and theory-sound way to deal with this reverse peeling process.
    '''
    
    
    # isolated walls (completely surrounded by other walls)
    len_walls_to_be_removed = len(walls_to_be_removed)
    walls_remaining = np.ones(len_walls_to_be_removed, dtype=np.bool_)

    
    try_again = True
    while np.sum(walls_remaining) != 0 and try_again:
        try_again = False
        
        for i0 in range(len_walls_to_be_removed):
            if walls_remaining[i0]:
                walls_to_be_removed_i = walls_to_be_removed[i0]
                [i,j,k] = walls_to_be_removed_i
                
                
                # We check if it neighbours at least two voids (otherwise it is a false wall).
                void_vals = check_neighbors_cube_single(fg, walls_to_be_removed_i, size, void_index_od, stop_after_2=True)
                no_voids = len(void_vals)
    
                
                if no_voids == 1:
                    # Success! One wall removed.
                    fg[i][j][k] = void_vals[0]
                    walls_remaining[i0] = False
                    try_again = True

                elif no_voids == 2:
                    # Success again! It is a "true" wall.
                    fg[i][j][k] = -2
                    walls_remaining[i0] = False
                    # but we can't try_again just from this, ofc
                    
                
                # else: maybe it will have a freshly converted void cell to attach to next round.
    
    
    # We only keep the walls we did not just remove, ofc.
    return fg, [list(_) for _ in walls_to_be_removed[walls_remaining]]

---
---
---